# Clairvoyant — DistilBERT Baseline (TODO-B)

Trains DistilBERT-base-uncased on ShareGPT, LMSYS, OASST1.
Evaluates pairwise ranking accuracy (Short vs Long pairs).
Results fill the `---` entries in `tab:baselines`.

**Runtime → T4 GPU**

In [ ]:
# Cell 1 — Install
!pip install -q transformers==4.40.0 torch scikit-learn pandas

In [ ]:
# Cell 2 — Upload data files
# Upload these 3 files from clairvoyant/data/:
#   training_data.csv   (ShareGPT, 6000 rows)
#   lmsys_labeled.csv   (LMSYS, 6000 rows)
#   oasst1_labeled.csv  (OASST1, ~1129 rows)
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 3 — Config
import torch, numpy as np, pandas as pd, json, time
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128
BATCH_SIZE = 64
EPOCHS     = 3
LR         = 2e-5
LABEL_BINS = [0, 200, 800, float('inf')]

DATASETS = {
    'ShareGPT': 'training_data.csv',
    'LMSYS':    'lmsys_labeled.csv',
    'OASST1':   'oasst1_labeled.csv',
}

print(f'Device: {DEVICE}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

In [ ]:
# Cell 4 — Dataset class + helpers
class PromptDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer):
        self.enc    = tokenizer(prompts, truncation=True, padding=True,
                                max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids':      self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels':         self.labels[i]}

def ranking_accuracy(p_long, y):
    idx_s = np.where(y == 0)[0]
    idx_l = np.where(y == 1)[0]
    if not len(idx_s) or not len(idx_l): return float('nan')
    correct = (p_long[idx_l][:, None] > p_long[idx_s][None, :]).sum()
    return float(correct) / (len(idx_l) * len(idx_s))

def run_dataset(name, csv_file):
    print(f'\n{"="*55}\n  {name}\n{"="*55}')
    df = pd.read_csv(csv_file)
    df = df[df['prompt'].notna() & (df['actual_output_tokens'] > 0)].copy()
    df['lbl3'] = pd.cut(df['actual_output_tokens'],
                        bins=LABEL_BINS, labels=[0,1,2], right=False).astype(int)
    df = df[df['lbl3'] != 1].copy()               # Short + Long only
    df['label'] = (df['lbl3'] == 2).astype(int)   # Long=1
    print(f'  Short: {(df.label==0).sum():,}  Long: {(df.label==1).sum():,}')

    tr, te = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])
    print(f'  Train: {len(tr):,}  Test: {len(te):,}')

    train_dl = DataLoader(
        PromptDataset(tr['prompt'].tolist(), tr['label'].tolist(), tokenizer),
        batch_size=BATCH_SIZE, shuffle=True)
    test_dl = DataLoader(
        PromptDataset(te['prompt'].tolist(), te['label'].tolist(), tokenizer),
        batch_size=BATCH_SIZE)

    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(train_dl) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, total//10, total)

    for epoch in range(1, EPOCHS+1):
        model.train(); loss_sum = 0; t0 = time.time()
        for b in train_dl:
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbl  = b['labels'].to(DEVICE)
            opt.zero_grad()
            out  = model(input_ids=ids, attention_mask=mask, labels=lbl)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            loss_sum += out.loss.item()
        print(f'  Epoch {epoch}/{EPOCHS}  loss={loss_sum/len(train_dl):.4f}  ({time.time()-t0:.0f}s)')

    model.eval(); probs, labs = [], []
    with torch.no_grad():
        for b in test_dl:
            out = model(input_ids=b['input_ids'].to(DEVICE),
                        attention_mask=b['attention_mask'].to(DEVICE))
            probs.extend(torch.softmax(out.logits, -1)[:,1].cpu().numpy())
            labs.extend(b['labels'].numpy())

    p, y = np.array(probs), np.array(labs)
    ra   = ranking_accuracy(p, y)
    cls  = (p >= 0.5).astype(int)
    print(f'  Ranking accuracy   : {ra*100:.1f}%')
    print(f'  Classification acc : {(cls==y).mean()*100:.1f}%')
    return {'dataset': name, 'ranking_accuracy_pct': round(ra*100,1),
            'classification_acc_pct': round((cls==y).mean()*100,1),
            'n_train': len(tr), 'n_test': len(te)}

print('Functions defined.')

In [ ]:
# Cell 5 — Run all datasets
import os
results = []
for name, csv_file in DATASETS.items():
    if os.path.exists(csv_file):
        results.append(run_dataset(name, csv_file))
    else:
        print(f'[SKIP] {name} — {csv_file} not uploaded')

print(f'\n{"="*55}')
print('  RESULTS — paste into tab:baselines')
print(f'{"="*55}')
for r in results:
    print(f"  DistilBERT  {r['dataset']:<12}  {r['ranking_accuracy_pct']:.1f}%")
print('\nP99 latency (865ms from §3.3) applies across all datasets.')

In [ ]:
# Cell 6 — Download results JSON
with open('distilbert_baseline.json', 'w') as f:
    json.dump(results, f, indent=2)
files.download('distilbert_baseline.json')
print('Downloaded.')